# ASII Direction Prediction (Up/Down) with Sentiment -- Sentiment-Covered Window Only

Combines two things already proven separately:
- **`Stock Direction Pred.ipynb`**'s method: predict up/down instead of exact price, with persistence and majority-class baselines.
- **`UNTR Sentiment Window Model.ipynb`**'s fix for sentiment's data-coverage problem: restrict train/val/test to only the dates where sentiment actually exists, instead of a ~1-year window that's 98% zero for the sentiment columns.

Sentiment source here is `data/news/ASII/asii_news_labeled.csv` (from `scripts/fetch_news.py`) -- a 3-class label (`positive`/`neutral`/`negative`) from a shareholder-perspective prompt, different from the UNTR notebooks' 2-class (`positive`/`negative`/`unclear`) OpenAI classifier. Mapped the same way: `positive=+1`, `negative=-1`, `neutral=0`.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

## Load price data

In [2]:
TICKER = "ASII"

df = pd.read_csv(f"../data/prices/{TICKER}.csv")
df_close = df[["Date", "Close"]].copy()
df_close.index = pd.to_datetime(df_close["Date"], format="%Y-%m-%d")
df_close = df_close.drop(columns="Date")

print(f"{TICKER}: {df_close.index.min().date()} to {df_close.index.max().date()} ({len(df_close)} trading days)")

ASII: 2025-09-26 to 2026-09-25 (244 trading days)


## Load sentiment and aggregate to daily

In [3]:
news = pd.read_csv("../data/news/ASII/asii_news_labeled.csv")
news["date"] = pd.to_datetime(news["timestamp"]).dt.normalize()

# neutral contributes 0 rather than being dropped -- "no clear signal that
# day" is itself informative, same convention as the UNTR notebooks' unclear/error.
sentiment_map = {"positive": 1, "negative": -1, "neutral": 0}
news["sentiment_num"] = news["label"].map(sentiment_map).fillna(0)

daily_sentiment = news.groupby("date").agg(
    sentiment_sum=("sentiment_num", "sum"),
    article_count=("sentiment_num", "count"),
).sort_index()

print(f"Labeled articles: {len(news)}, {news['date'].min().date()} to {news['date'].max().date()}")
print(f"Unique calendar dates with news: {len(daily_sentiment)}")

Labeled articles: 248, 2026-05-30 to 2026-09-25
Unique calendar dates with news: 85


## Restrict to the sentiment-covered window

Same approach as `UNTR Sentiment Window Model.ipynb`: reindex onto the trading-day index (weekend/holiday news has no matching trading day and is dropped, no next-session rollover), then slice `df_close` down to only the days from the first day with any coverage.

In [4]:
daily_sentiment = daily_sentiment.reindex(df_close.index, fill_value=0)
df_close["sentiment_sum"] = daily_sentiment["sentiment_sum"]
df_close["article_count"] = daily_sentiment["article_count"]

news_window_start = df_close.index[df_close["article_count"] > 0].min()
df_close = df_close.loc[df_close.index >= news_window_start].copy()

print(f"Sentiment-covered window: {df_close.index.min().date()} to {df_close.index.max().date()} ({len(df_close)} trading days)")
print(f"Trading days with any news in this window: {(df_close['article_count'] > 0).sum()} / {len(df_close)}")

Sentiment-covered window: 2026-06-02 to 2026-09-25 (82 trading days)
Trading days with any news in this window: 71 / 82


## Feature engineering: return/lag features + sentiment + binary direction target

In [5]:
df_close["dayofweek"] = df_close.index.dayofweek

df_close["ret"] = df_close["Close"].pct_change()
df_close["ret_lag1"] = df_close["ret"].shift(1)
df_close["ret_lag2"] = df_close["ret"].shift(2)
df_close["ret_lag3"] = df_close["ret"].shift(3)
df_close["ret_lag5"] = df_close["ret"].shift(5)
df_close["vol_5"] = df_close["ret"].shift(1).rolling(5).std()

# Sentiment features -- shift(1) so today's row never sees today's own news
df_close["sentiment_lag1"] = df_close["sentiment_sum"].shift(1)
df_close["article_count_lag1"] = df_close["article_count"].shift(1)
df_close["sentiment_roll3"] = df_close["sentiment_sum"].shift(1).rolling(3).sum()

# Binary target: did the close go UP that day?
df_close["direction"] = (df_close["ret"] > 0).astype(int)

FEATURES = ["ret_lag1", "ret_lag2", "ret_lag3", "ret_lag5", "vol_5", "dayofweek"]
SENT_FEATURES = ["sentiment_lag1", "article_count_lag1", "sentiment_roll3"]
FEATURES_SENT = FEATURES + SENT_FEATURES
TARGET = "direction"

df_close[["Close"] + FEATURES_SENT + [TARGET]].tail()

,Close,ret_lag1,ret_lag2,ret_lag3,ret_lag5,vol_5,dayofweek,sentiment_lag1,article_count_lag1,sentiment_roll3,direction
Date,,,,,,,,,,,
2026-09-21,4820.0,-0.016327,0.010309,-0.008180,0.014257,0.015039,0,0.0,1.0,0.0,0
2026-09-22,4780.0,0.000000,-0.016327,0.010309,-0.018072,0.011812,1,-1.0,1.0,-1.0,0
2026-09-23,4750.0,-0.008299,0.000000,-0.016327,-0.008180,0.010092,2,0.0,1.0,-1.0,0
2026-09-24,4770.0,-0.006276,-0.008299,0.000000,0.010309,0.009954,3,6.0,14.0,5.0,1
2026-09-25,4690.0,0.004211,-0.006276,-0.008299,-0.016327,0.007907,4,2.0,4.0,8.0,0


## Train / validation / test split

Chronological, within the sentiment-covered window only. Small sample by construction -- treat any result here as a first, low-power read, not a confident verdict.

In [6]:
N_TEST = 20
N_VAL = 15

data = df_close.dropna(subset=FEATURES_SENT + [TARGET])
test = data.iloc[-N_TEST:]
train_full = data.iloc[:-N_TEST]
val = train_full.iloc[-N_VAL:]
train = train_full.iloc[:-N_VAL]

print(f"train / val / test : {len(train)} / {len(val)} / {len(test)}")
print(f"test window        : {test.index.min().date()} to {test.index.max().date()}")
print(f"train up-day rate  : {train[TARGET].mean():.1%}")
print(f"test up-day rate   : {test[TARGET].mean():.1%}")

train / val / test : 41 / 15 / 20
test window        : 2026-08-31 to 2026-09-25
train up-day rate  : 41.5%
test up-day rate   : 45.0%


## Baselines: what are we trying to beat?

Persistence (tomorrow = today's direction) and majority-class (always guess train's more common direction) -- same as `Stock Direction Pred.ipynb`. A model that can't beat both isn't adding anything.

In [7]:
persistence_pred = df_close[TARGET].shift(1).loc[test.index]
persistence_acc = accuracy_score(test[TARGET], persistence_pred)

majority_class = train[TARGET].mode()[0]
majority_pred = np.full(len(test), majority_class)
majority_acc = accuracy_score(test[TARGET], majority_pred)

print(f"Persistence baseline accuracy   : {persistence_acc:.1%}")
print(f"Majority-class baseline accuracy: {majority_acc:.1%}  (always predicts {'UP' if majority_class == 1 else 'DOWN'})")

Persistence baseline accuracy   : 35.0%
Majority-class baseline accuracy: 55.0%  (always predicts DOWN)


## Train and score: with vs. without sentiment

In [8]:
def fit_and_score(features, label):
    X_tr, y_tr = train[features], train[TARGET]
    X_val, y_val = val[features], val[TARGET]
    X_te, y_te = test[features], test[TARGET]

    clf = xgb.XGBClassifier(
        n_estimators=1000,
        early_stopping_rounds=50,
        learning_rate=0.01,
        max_depth=3,
        eval_metric="logloss",
    )
    clf.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], verbose=False)

    pred = clf.predict(X_te)
    proba = clf.predict_proba(X_te)[:, 1]
    acc = accuracy_score(y_te, pred)
    auc = roc_auc_score(y_te, proba)
    fi = pd.Series(clf.feature_importances_, index=clf.feature_names_in_).sort_values(ascending=False)

    print(f"[{label}] accuracy: {acc:.1%}  |  ROC-AUC: {auc:.3f}  |  best_iteration: {clf.best_iteration}")
    return clf, acc, auc, fi, pred


clf_base, acc_base, auc_base, fi_base, pred_base = fit_and_score(FEATURES, "no sentiment")
clf_sent, acc_sent, auc_sent, fi_sent, pred_sent = fit_and_score(FEATURES_SENT, "with sentiment")

print()
print(f"Persistence baseline accuracy   : {persistence_acc:.1%}")
print(f"Majority-class baseline accuracy: {majority_acc:.1%}")
print(f"XGBoost, no sentiment           : {acc_base:.1%}  (ROC-AUC {auc_base:.3f})")
print(f"XGBoost, with sentiment         : {acc_sent:.1%}  (ROC-AUC {auc_sent:.3f})")

print("\nConfusion matrix, with sentiment (rows=actual, cols=predicted; 0=down, 1=up):")
print(confusion_matrix(test[TARGET], pred_sent))

print("\nFeature importances (with sentiment):")
print(fi_sent)

[no sentiment] accuracy: 55.0%  |  ROC-AUC: 0.773  |  best_iteration: 22
[with sentiment] accuracy: 55.0%  |  ROC-AUC: 0.707  |  best_iteration: 38

Persistence baseline accuracy   : 35.0%
Majority-class baseline accuracy: 55.0%
XGBoost, no sentiment           : 55.0%  (ROC-AUC 0.773)
XGBoost, with sentiment         : 55.0%  (ROC-AUC 0.707)

Confusion matrix, with sentiment (rows=actual, cols=predicted; 0=down, 1=up):
[[10  1]
 [ 8  1]]

Feature importances (with sentiment):
ret_lag1              0.188487
ret_lag3              0.170814
ret_lag2              0.137014
vol_5                 0.122682
sentiment_roll3       0.103971
ret_lag5              0.102847
dayofweek             0.093268
article_count_lag1    0.080916
sentiment_lag1        0.000000
dtype: float32


## Reading this result

Actual run: **train/val/test = 41/15/20 days**, sentiment coverage 71/82 days (86%) -- by far the densest window we've had (UNTR's equivalent was 50%). Both models genuinely trained this time (`best_iteration` 22 and 38, not 0), so this isn't another "not enough data to learn anything" result -- the sentiment model had real signal to fit.

**Persistence 35.0%, majority-class (always DOWN) 55.0%, no-sentiment model 55.0% (ROC-AUC 0.773), with-sentiment model 55.0% (ROC-AUC 0.707).** Same accuracy as the no-sentiment model and the majority-class baseline, but a *lower* ROC-AUC -- and the confusion matrix shows why: the sentiment model predicted UP only twice in 20 days (1 right, 1 wrong), landing close to "always guess down" rather than genuinely discriminating. `sentiment_roll3` and `article_count_lag1` did earn real feature importance (10.4% and 8.1%) this time, unlike every prior UNTR run where sentiment sat at exactly 0 -- so the model *did* use the signal, it just didn't help rank days correctly on this particular 20-day test.

Don't over-read this either direction: it's one 20-day test slice, and "sentiment features got used but the model got slightly worse" is a genuinely different failure mode than "sentiment was structurally invisible to the model" (UNTR's result). Worth another look once more days accumulate (`scripts/fetch_news.py` -> rerun this notebook) rather than either dismissing sentiment for ASII or trusting this one run.